In [ ]:
import pandas as pd
import openai
import json
import re
import ast
import numpy as np
from tqdm import tqdm

tqdm.pandas()

import os
from dotenv import load_dotenv

load_dotenv()
openai.api_key = os.getenv("OPENAI_API_KEY")

In [3]:

model_names = ['xml_roberta', 'sbert', 'ukr-roberta']

def parse_textfooler_results(filenames, model_names):
    all_dfs = []

    # regex to split each “text N” block
    entry_splitter = re.compile(r'(?=^text \d+)', re.M)

    for fname, model in zip(filenames, model_names):
        with open(fname, 'r', encoding='utf-8') as f:
            content = f.read()

        # isolate each entry
        entries = entry_splitter.split(content)
        records = []

        for entry in entries:
            entry = entry.strip()
            if not entry:
                continue

            # capture orig label + all text up to next "adv sent"
            orig_m = re.search(
                r'orig sent \((\d+)\):\s*(.*?)\n(?=adv sent)',
                entry,
                re.S
            )
            # capture adv label + all text up to "Replacements"
            adv_m = re.search(
                r'adv sent \((\d+)\):\s*(.*?)\n(?=Replacements)',
                entry,
                re.S
            )
            # capture the replacements list
            repl_m = re.search(
                r'Replacements\s+(\[.*?\])',
                entry,
                re.S
            )

            if orig_m and adv_m and repl_m:
                orig_label   = int(orig_m.group(1))
                orig_text    = orig_m.group(2).strip()
                adv_label    = int(adv_m.group(1))
                adv_text     = adv_m.group(2).strip()
                replacements = ast.literal_eval(repl_m.group(1))

                records.append({
                    "original_text":      orig_text,
                    "adversarial_text":   adv_text,
                    "original_label":     orig_label,
                    "adversarial_label":  adv_label,
                    "all_replacements":   replacements
                })

        df = pd.DataFrame(records)
        df['model'] = model
        print(f"For {model!r} there are {len(df)} samples")
        all_dfs.append(df)

    result = pd.concat(all_dfs, ignore_index=True)
    print("Total rows:", result.shape[0])
    return result

def parse_bert_attack_results(filenames, model_names):

    all_dfs = []

    for fname, model in zip(filenames, model_names):
        with open(fname, 'r', encoding='utf-8') as f:
            data = json.load(f)

        df = pd.DataFrame(data)
        df = df[df.success==4]
        df['model'] = model
        print(f"For model {model!r}, parsed {len(df)} substitutions")
        all_dfs.append(df)

    result = pd.concat(all_dfs, ignore_index=True)
    result.drop(columns=['success', 'num_word', 'query'], inplace = True)
    result.columns = ['original_label', 'all_replacements', 'adversarial_label', 'original_text', 'adversarial_text', 'model']

    print("Total substitution records:", result.shape[0])
    return result

def sample_per_model(df, n=50, random_state = 42):
    sampled = (
        df
        .groupby('model', group_keys=False)
        .apply(lambda grp: grp.sample(n=min(len(grp), n), random_state=random_state))
        .reset_index(drop=True)
    )
    return sampled

def prepare_dataset_sample(files_text_fooler, filer_bert_attack, num=200):
    df_textfooler = parse_textfooler_results(files_text_fooler, model_names)
    sample_textfooler = sample_per_model(df_textfooler, num)
    sample_textfooler['attack'] = 'textfooler'

    df_bertattack = parse_bert_attack_results(filer_bert_attack, model_names)
    sample_bert_attack = sample_per_model(df_bertattack, num)
    sample_bert_attack['attack'] = 'bertattack'

    return pd.concat([sample_textfooler, sample_bert_attack])

In [4]:
filenames_reviews_textfooler = ['/home/mudryi/phd_projects/textfooler_ukr/adv_results_reviews_xml_roberta/adversaries.txt',
                     '/home/mudryi/phd_projects/textfooler_ukr/adv_results_reviews_sentence_transformer/adversaries.txt',
                     '/home/mudryi/phd_projects/textfooler_ukr/adv_results_reviews_ukr-roberta/adversaries.txt']

filenames_reviews_bertattack = ['/home/mudryi/phd_projects/bert_attack_uk/adv_results_reviews_xml_roberta',
                    '/home/mudryi/phd_projects/bert_attack_uk/adv_results_reviews_sentence-transformers',
                    "/home/mudryi/phd_projects/bert_attack_uk/adv_results_reviews_youscan"]

df_reviews = prepare_dataset_sample(filenames_reviews_textfooler, filenames_reviews_bertattack)
df_reviews.head()

For 'xml_roberta' there are 3153 samples
For 'sbert' there are 3538 samples
For 'ukr-roberta' there are 4182 samples
Total rows: 10873
For model 'xml_roberta', parsed 1139 substitutions
For model 'sbert', parsed 1239 substitutions
For model 'ukr-roberta', parsed 1227 substitutions
Total substitution records: 3605


/tmp/ipykernel_3522/668039602.py:90: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda grp: grp.sample(n=min(len(grp), n), random_state=random_state))
/tmp/ipykernel_3522/668039602.py:90: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda grp: grp.sample(n=min(len(grp), n), random_state=random_state))


,original_text,adversarial_text,original_label,adversarial_label,all_replacements,model,attack
0,"Дуже стійкий, дуже «чоловічий» і дуже статусни...","Дуже неподатливий, дуже «чоловічий» і дуже ста...",4.0,2.0,"[(стійкий, неподатливий, 2), (чудове, прегарни...",sbert,textfooler
1,За таку ціну з No Frost не знайти,За таку ціну з No Frost не установлювати,4.0,0.0,"[(знайти, установлювати, 14)]",sbert,textfooler
2,Смакота. Насичений чай з дуже приємним смаком....,Смакота. Сильний чай з дуже поманливим смаком....,4.0,3.0,"[(приємним, поманливий, 11), (чудово, добре, 5...",sbert,textfooler
3,Дуже довго вибирали пилосос і нарешті зупинили...,Дуже довго вирили пилосос і нарешті спинились ...,4.0,0.0,"[(рекомендую, відрекомендовувати, 146), (має, ...",sbert,textfooler
4,"Товар дуже сподобався, краще мишки я не зустрі...","Товар дуже пригледівсь, краще мишки я не зустр...",2.0,1.0,"[(сподобався, пригледітися, 4)]",sbert,textfooler


In [5]:
filenames_news_textfooler = ['/home/mudryi/phd_projects/textfooler_ukr/adv_results_news_xlm-roberta-base/adversaries.txt',
                  '/home/mudryi/phd_projects/textfooler_ukr/adv_results_news_sentence-transformers/adversaries.txt',
                  '/home/mudryi/phd_projects/textfooler_ukr/adv_results_news_youscan/adversaries.txt']

filenames_news_bert_attack = ["/home/mudryi/phd_projects/bert_attack_uk/adv_results_news_xlm-roberta-base",
                  "/home/mudryi/phd_projects/bert_attack_uk/adv_results_news_sentence-transformers",
                  "/home/mudryi/phd_projects/bert_attack_uk/adv_results_news_youscan"]

df_news =prepare_dataset_sample(filenames_news_textfooler, filenames_news_bert_attack)

label_list = ['бізнес', 'новини', 'політика', 'спорт', 'технології']
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}
df_news['original_label'] = df_news['original_label'].map(id2label)
df_news.head()

For 'xml_roberta' there are 1394 samples
For 'sbert' there are 1419 samples
For 'ukr-roberta' there are 1838 samples
Total rows: 4651
For model 'xml_roberta', parsed 2098 substitutions
For model 'sbert', parsed 2233 substitutions
For model 'ukr-roberta', parsed 1706 substitutions
Total substitution records: 6037


/tmp/ipykernel_3522/668039602.py:90: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda grp: grp.sample(n=min(len(grp), n), random_state=random_state))
/tmp/ipykernel_3522/668039602.py:90: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda grp: grp.sample(n=min(len(grp), n), random_state=random_state))


,original_text,adversarial_text,original_label,adversarial_label,all_replacements,model,attack
0,Колаборантка Поклонська зібралась випускати то...,Колаборантка Поклонська згромаджувалась випуск...,новини,2,"[(зібралась, згромаджуватися, 5)]",sbert,textfooler
1,Арабську мову хочуть зробити обов'язковою для ...,Арабську мову охотяться зробити обов'язковою д...,політика,4,"[(школах, шкілка, 17), (хочуть, охотитися, 5)]",sbert,textfooler
2,"Хлопчик хотів змити себе в унітаз, щоб опинити...","Хлопчик хотів омити себе в унітаз, щоб опинити...",новини,3,"[(змити, омити, 5)]",sbert,textfooler
3,Слабкий долар дає надію,Хирлявий долар дає сподіванку,бізнес,1,"[(Слабкий, хирлявий, 0), (надію, сподіванка, 6)]",sbert,textfooler
4,Що допоможе збільшити потік вантажів через пор...,Що допоможе пожвавити струмок вантажів через п...,бізнес,1,"[(збільшити, пожвавити, 4), (потік, струмок, 6)]",sbert,textfooler


In [6]:
filenames_unlp_textfooler = ['/home/mudryi/phd_projects/textfooler_ukr/adv_results_unlp_xlm-roberta-base/adversaries.txt',
                  '/home/mudryi/phd_projects/textfooler_ukr/adv_results_unlp_sentence-transformers/adversaries.txt',
                  '/home/mudryi/phd_projects/textfooler_ukr/adv_results_unlp_youscan/adversaries.txt']

filenames_unlp_bertattack = ['/home/mudryi/phd_projects/bert_attack_uk/adv_results_unlp_xlm-roberta-base',
                  '/home/mudryi/phd_projects/bert_attack_uk/adv_results_unlp_sentence-transformers',
                  "/home/mudryi/phd_projects/bert_attack_uk/adv_results_unlp_youscan"]

df_unlp =prepare_dataset_sample(filenames_unlp_textfooler, filenames_unlp_bertattack)
df_unlp.head()

For 'xml_roberta' there are 152 samples
For 'sbert' there are 161 samples
For 'ukr-roberta' there are 167 samples
Total rows: 480
For model 'xml_roberta', parsed 91 substitutions
For model 'sbert', parsed 92 substitutions
For model 'ukr-roberta', parsed 88 substitutions
Total substitution records: 271


/tmp/ipykernel_3522/668039602.py:90: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda grp: grp.sample(n=min(len(grp), n), random_state=random_state))
/tmp/ipykernel_3522/668039602.py:90: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda grp: grp.sample(n=min(len(grp), n), random_state=random_state))


,original_text,adversarial_text,original_label,adversarial_label,all_replacements,model,attack
0,Присутність росіян на окупованій ЗАЕС – ключов...,Присутність росіян на окупованій ЗАЕС – ключов...,1,0,"[(ядерної, отакенний, 18), (літні, злітний, 111)]",sbert,textfooler
1,⚡️\n5 листопада 2023 року.\nУраження ударних б...,⚡️\n5 падолисту 2023 року.\nУраження ударних б...,0,1,"[(Південь, зюйд, 59), (листопада, падолист, 4)...",sbert,textfooler
2,Нацполіція почала передавати в Інтерпол дані п...,Нацполіція почала передавати в Інтерпол дані п...,0,1,"[(внесено, приплюсовувати, 76), (повернути, кр...",sbert,textfooler
3,Деніел Вінні Свіфт – американський доброволець...,Деніел Вінні Свіфт – американський доброволець...,1,0,"[(котик, кіт, 69), (відправляється, виряджатис...",sbert,textfooler
4,✅\n✅\n✅\n✅\n🛵\n🛵\n🛵\n🛵\n БпЛА у КРЕМЕНЕЦЬКОМУ ...,✅\n✅\n✅\n✅\n🛵\n🛵\n🛵\n🛵\n БпЛА у КРЕМЕНЕЦЬКОМУ ...,0,1,"[(північний, опівнічний, 71), (груп, стадо, 54)]",sbert,textfooler


# GPT news

In [149]:
system_prompt = (
    "Ви — модель GPT‑4o, мета якої — класифікувати українські заголовки новин "
    "за однією із п’яти категорій: «бізнес», «новини», «політика», «спорт», «технології». "
    "Поверніть тільки назву категорії."
)

few_shot_examples = (
    "Заголовок: \"Уряд затвердив нову стратегію економічного розвитку\"\n"
    "Категорія: політика\n\n"
    "Заголовок: \"Apple анонсує новий iPhone з поліпшеною камерою\"\n"
    "Категорія: технології\n\n"
    "Заголовок: \"Шахтар перемагає у фіналі Ліги чемпіонів\"\n"
    "Категорія: спорт"
)

# 4. Classification function
def classify_headline(headline: str) -> str:
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": few_shot_examples},
        {"role": "user",   "content": f"Заголовок: \"{headline}\"\nКатегорія:"}
    ]
    response = openai.chat.completions.create(
        model="gpt-4o",
        messages=messages,
        temperature=0.0,
        max_tokens=10
    )
    return response.choices[0].message.content.strip()

In [150]:
df_news["gpt_orig_pred"] = df_news["original_text"].progress_apply(classify_headline)
df_news["gpt_adv_pred"] = df_news["adversarial_text"].progress_apply(classify_headline)

100%|██████████| 1200/1200 [12:11<00:00,  1.64it/s]


In [151]:
np.mean(df_news['original_label'] == df_news["gpt_orig_pred"]), sum(df_news['original_label'] == df_news["gpt_orig_pred"])

(np.float64(0.6233333333333333), 748)

In [165]:
np.mean(df_news['original_label'] == df_news["gpt_adv_pred"]), sum(df_news['original_label'] == df_news["gpt_adv_pred"])

(np.float64(0.565), 678)

In [9]:
df_news = pd.read_pickle('gpt_results_news_1200.pickle')

In [12]:
# df_news.groupby('attack').apply(lambda x: np.mean(x['original_label'] == x["gpt_orig_pred"]))
df_news.groupby('attack').apply(lambda x: np.mean(x['original_label'] == x["gpt_adv_pred"]))

/tmp/ipykernel_2713/1477796404.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_news.groupby('attack').apply(lambda x: np.mean(x['original_label'] == x["gpt_adv_pred"]))


attack
bertattack    0.52
textfooler    0.61
dtype: float64

In [153]:
df_news.to_pickle('gpt_results_news_1200.pickle')

# GPT reviews

In [154]:
system_prompt = (
    "Ви — модель GPT‑4o, мета якої — оцінити якість відгуку українською мовою "
    "за шкалою від 0 до 4, де:\n"
    "0 — дуже погано\n"
    "1 — погано\n"
    "2 — посередньо\n"
    "3 — добре\n"
    "4 — дуже добре\n"
    "Поверніть **тільки** JSON-об’єкт із ключем \"predicted_label\" "
    "без будь‑яких трикрапок чи пояснень."
)

# 4. Few-shot examples as plain JSON
few_shot_examples = (
    "Приклад 1:\n"
    "Вхід: {\"review\": \"Я замовив доставку вчасно, але піца була холодною й пересоленою.\"}\n"
    "Вихід: {\"predicted_label\": 1}\n\n"
    "Приклад 2:\n"
    "Вхід: {\"review\": \"Чудовий сервіс, ввічливий персонал і дуже смачна їжа!\"}\n"
    "Вихід: {\"predicted_label\": 4}\n\n"
    "Приклад 3:\n"
    "Вхід: {\"review\": \"Загалом непогано, але десерт міг бути солодшим.\"}\n"
    "Вихід: {\"predicted_label\": 2}\n\n"
    "Приклад 4:\n"
    "Вхід: {\"review\": \"Не рекомендую — замовлення загубили, потім переплутали страви.\"}\n"
    "Вихід: {\"predicted_label\": 0}"
)

# 5. Updated classification function with stop sequence and higher max_tokens
def classify_review(review: str) -> str:
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": few_shot_examples},
        {"role": "user",   "content": f"Вхід: {{\"review\": \"{review}\"}}\nВихід:"}
    ]
    response = openai.chat.completions.create(
        model="gpt-4o",
        messages=messages,
        temperature=0.0,
        max_tokens=8,
        stop=["}"]
    )
    # Append closing brace if missing
    content = response.choices[0].message.content.strip()
    return content + "}" if not content.endswith("}") else content


In [155]:
df_reviews["gpt_orig_pred"] = df_reviews["original_text"].progress_apply(classify_review)
df_reviews["gpt_orig_pred"] = df_reviews["gpt_orig_pred"].apply(lambda x: json.loads(x)["predicted_label"])

df_reviews["gpt_adv_pred"] = df_reviews["adversarial_text"].progress_apply(classify_review)
df_reviews["gpt_adv_pred"] = df_reviews["gpt_adv_pred"].apply(lambda x: json.loads(x)["predicted_label"])


  0%|          | 0/1200 [00:00<?, ?it/s]

100%|██████████| 1200/1200 [15:43<00:00,  1.27it/s]


In [156]:
np.mean(df_reviews['original_label'] == df_reviews["gpt_orig_pred"]), sum(df_reviews['original_label'] == df_reviews["gpt_orig_pred"])

(np.float64(0.3641666666666667), 437)

In [157]:
np.mean(df_reviews['original_label'] == df_reviews["gpt_adv_pred"]), sum(df_reviews['original_label'] == df_reviews["gpt_adv_pred"])

(np.float64(0.25), 300)

In [158]:
df_reviews.to_pickle('gpt_results_reviews_1200.pickle')

In [4]:
df_reviews = pd.read_pickle('gpt_results_reviews_1200.pickle')

In [8]:
# df_reviews.groupby('attack').apply(lambda x: np.mean(x['original_label'] == x["gpt_orig_pred"]))
df_reviews.groupby('attack').apply(lambda x: np.mean(x['original_label'] == x["gpt_adv_pred"]))

/tmp/ipykernel_2713/3229721215.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_reviews.groupby('attack').apply(lambda x: np.mean(x['original_label'] == x["gpt_adv_pred"]))


attack
bertattack    0.21
textfooler    0.29
dtype: float64

In [7]:
df_reviews.attack.value_counts()

attack
textfooler    600
bertattack    600
Name: count, dtype: int64

# GPT UNLP

In [159]:
system_prompt = (
    "Ви — модель GPT‑4o, мета якої — визначити, чи містить український текст у соціальних мережах "
    "маніпулятивні риторичні чи стилістичні прийоми, спрямовані вплинути на аудиторію без чітких фактів. "
    "Поверніть лише JSON-об’єкт із ключем \"predicted_label\":\n"
    "1 — якщо маніпуляція є,\n"
    "0 — якщо маніпуляції немає."
)


# 4. Few-shot examples as plain JSON
few_shot_examples = (
    # Example 1: маніпуляція (емоційне звернення без доказів)
    "Вхід: \"Всі нормальні люди вже бачать правду! Приєднуйтеся і ви, поки вам не пізно!\"\n"
    "Вихід: {\"predicted_label\": 1}\n\n"
    # Example 2: немає маніпуляції (фактичний опис)
    "Вхід: \"Згідно з офіційним звітом, кількість відвідувачів музею зросла на 15%.\"\n"
    "Вихід: {\"predicted_label\": 0}\n\n"
    # Example 3: маніпуляція (генералізація + страх)
    "Вхід: \"Уряд мовчить про реальні витрати — вони приховують від вас правду!\"\n"
    "Вихід: {\"predicted_label\": 1}\n\n"
    # Example 4: немає маніпуляції (нейтральна порада)
    "Вхід: \"Не забудьте перевірити рівень масла перед довгою поїздкою.\"\n"
    "Вихід: {\"predicted_label\": 0}"
)


# 5. Updated classification function with stop sequence and higher max_tokens
def classify_review(review: str) -> str:
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": few_shot_examples},
        {"role": "user",   "content": f"Вхід: {{\"text\": \"{review}\"}}\nВихід:"}
    ]
    response = openai.chat.completions.create(
        model="gpt-4o",
        messages=messages,
        temperature=0.0,
        max_tokens=8,
        stop=["}"]
    )
    # Append closing brace if missing
    content = response.choices[0].message.content.strip()
    return content + "}" if not content.endswith("}") else content


In [160]:
df_unlp["gpt_orig_pred"] = df_unlp["original_text"].progress_apply(classify_review)
df_unlp["gpt_orig_pred"] = df_unlp["gpt_orig_pred"].apply(lambda x: json.loads(x)["predicted_label"])

df_unlp["gpt_adv_pred"] = df_unlp["adversarial_text"].progress_apply(classify_review)
df_unlp["gpt_adv_pred"] = df_unlp["gpt_adv_pred"].apply(lambda x: json.loads(x)["predicted_label"])

  0%|          | 0/751 [00:00<?, ?it/s]

100%|██████████| 751/751 [12:39<00:00,  1.01s/it]


In [161]:
np.mean(df_unlp['original_label'] == df_unlp["gpt_orig_pred"]), sum(df_unlp['original_label'] == df_unlp["gpt_orig_pred"])

(np.float64(0.7709720372836218), 579)

In [162]:
np.mean(df_unlp['original_label'] == df_unlp["gpt_adv_pred"]), sum(df_unlp['original_label'] == df_unlp["gpt_adv_pred"])

(np.float64(0.6990679094540613), 525)

In [163]:
df_unlp.to_pickle('gpt_results_unlp_1200.pickle')

In [ ]:
df_news = pd.read_pickle('gpt_results_news_1200.pickle')

In [181]:
# df_unlp.groupby('attack').apply(lambda x: np.mean(x['original_label'] == x["gpt_orig_pred"]))
df_unlp.groupby('attack').apply(lambda x: np.mean(x['original_label'] == x["gpt_adv_pred"]))

/tmp/ipykernel_662968/1523333618.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_unlp.groupby('attack').apply(lambda x: np.mean(x['original_label'] == x["gpt_adv_pred"]))


attack
bertattack    0.642066
textfooler    0.731250
dtype: float64